# Fixed-τ Homogeneous Sweep + Min/Max O-Information (Colab)

**Supervisor feedback, June 2026 (Marcus Ghosh + Hardik Rajpal).**

We already know het beats hom and that *both learn solutions around ~25 ms*. The open question:
**is het's advantage the *heterogeneity* of time constants, or simply that het has *some longer* τ?**
— plus a sharper way to read O-information.

This single notebook does both halves:

### Part 1 — train fixed-τ homogeneous controls (needs GPU)
For each of the **10 seeds**, retrain a homogeneous network with the hidden τ_mem **frozen** at
**15 ms** and (separately) **35 ms**. Each control is **paired** to its het seed: identical initial weights,
identical data, identical optimiser / 50-epoch schedule — the *only* difference from the existing nets is the
single fixed τ. → **20 new checkpoints.** Together with the existing het and hom(~25 ms learned) nets this
gives the τ-ladder **15 / 25 / 35 ms + het** that Marcus asked for.

### Part 2 — min / max O-Information (Hardik)
Average O across subsets is the wrong statistic (dense RNNs are redundant by default). Per (network, k) we
report **min O** (most synergistic subset), **max O** (most redundant), and the **spread (max − min)** — how
far heterogeneity moves the extremes — across **het / hom-15 / hom-25 / hom-35**. Plus the het scatter:
x = mean pairwise **|Δτ|** of a subset's neurons, y = its O-info.

---
**Before running:** (1) put this folder on Google Drive; (2) Runtime → Change runtime type → **GPU**;
(3) set `PROJECT_ROOT` and `SHD_DIR` in the config cell; (4) Run all. Both parts are **resumable** —
finished checkpoints / spike caches / O-records are skipped.


## Part 1 · 1 — Colab setup & dependencies

In [ ]:
# ========================== 1. Colab setup ==========================
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # data_gen needs PyTables to read the SHD .h5 files
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tables"], check=False)
print("Colab:", IN_COLAB)

## Part 1 · 2 — Paths & imports

`seeded_runs_common` bundles the paired-training machinery; the SNN model code lives in `code/SuGD_code`.
We add both to `sys.path` *before* importing, then re-point the module's (Windows) paths at this folder so it
runs on Colab.

In [ ]:
# ========================== 2. Paths + imports ==========================
from pathlib import Path
import numpy as np, torch

# The whole 'research project (SNN Info Theory)' folder is at the Drive root, mirroring the local layout,
# so this notebook's folder is NESTED inside it (not directly under MyDrive).
# >>> EDIT only if your Drive folder name differs <<<
RESEARCH_ROOT = Path("/content/drive/MyDrive/research project (SNN Info Theory)")
PROJECT_ROOT  = RESEARCH_ROOT / "Fixed-Tau Sweep (Colab Upload)"   # this notebook's folder
SHD_DIR       = RESEARCH_ROOT / "data" / "shd"                     # holds shd_train.h5 / shd_test.h5

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
CODE   = PROJECT_ROOT / "code"
PAPER  = CODE / "SuGD_code"
REFCK  = PROJECT_ROOT / "reference_checkpoints"          # existing het + hom(~25 ms) nets (bundled)
OUTDIR = PROJECT_ROOT / "outputs"; OUTDIR.mkdir(exist_ok=True)
CKPT_OUT = OUTDIR / "checkpoints"; CKPT_OUT.mkdir(parents=True, exist_ok=True)

# code/ and code/SuGD_code must be importable BEFORE `import seeded_runs_common`
for p in (str(CODE), str(PAPER)):
    if p not in sys.path:
        sys.path.insert(0, p)

import seeded_runs_common as src

# Re-point the module's hard-coded Windows paths at this Colab folder
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
src.DEVICE          = DEVICE
src.PROJECT_ROOT    = PROJECT_ROOT
src.PROJECT_FILES   = PROJECT_ROOT
src.PAPER_ROOT      = PAPER
src.CHECKPOINT_ROOT = CKPT_OUT
src.TAU_ARTIFACT_PATH = PROJECT_ROOT / "Tau from 15 epoch run.json"
src.SHD_TRAIN       = SHD_DIR / "shd_train.h5"
src.SHD_TEST        = SHD_DIR / "shd_test.h5"
assert src.SHD_TRAIN.exists() and src.SHD_TEST.exists(), f"SHD .h5 not found under {SHD_DIR}"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

## Part 1 · 3 — Configuration (matches the original 64-neuron, 4-class, 50-epoch run)

These overrides reproduce the *exact* training recipe of the trained nets (`seeded_run_1_64n`), so the only
difference in the controls is the **frozen τ**.

In [ ]:
# ========================== 3. Config ==========================
SEEDS        = [101, 202, 210, 340, 440, 550, 660, 710, 820, 930]   # the 10 paired seeds
TASK_KEY     = "4class"
MEM_FAMILY   = "lognormal"
FIXED_TAUS_MS = [15.0, 35.0]            # the two new homogeneous controls (ms)

# --- reproduce the seeded_run_1_64n recipe exactly (patch BASE_PRMS before building models) ---
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
src.BASE_PRMS["nb_recurrent"]        = 64
src.BASE_PRMS["batch_size"]          = 256
src.BASE_PRMS["nb_epochs"]           = 50
src.BASE_PRMS["early_stop_patience"] = 10
src.BASE_PRMS["optimizer"]           = "adam"
src.BASE_PRMS["weight_decay"]        = 1e-4     # homogeneous nets use WD=1e-4 (as in the original hom)
src.BASE_PRMS["lr"]                  = 2e-3
src.BASE_PRMS["lr_ab"]               = 2e-3
src.BASE_PRMS["lr_patience"]         = 5
src.BASE_PRMS["lr_factor"]           = 0.5
src.BASE_PRMS["lr_min"]              = 5e-5
src.BASE_PRMS["compile_model"]       = False

# checkpoint filename templates
OUT_NAME = "4class_seed{s}_hom_fixed{tau}ms_lognormal.pt"    # the NEW controls (this notebook)
HET_NAME = "4class_seed{s}_het_lognormal.pt"                 # bundled reference (het)
HOM_NAME = "4class_seed{s}_hom_geommean_lognormal.pt"        # bundled reference (hom, learned ~25 ms)

print(f"Training {len(FIXED_TAUS_MS)} x {len(SEEDS)} = {len(FIXED_TAUS_MS)*len(SEEDS)} fixed-tau hom nets")
print(f"  nb_recurrent=64  epochs=50  bs=256  optim=adam  lr=2e-3  wd=1e-4  fixed taus={FIXED_TAUS_MS} ms")
print(f"  output -> {CKPT_OUT}")

## Part 1 · 4 — Build a frozen-τ homogeneous net (paired to its het seed)

Identical to the original hom **at initialisation** (non-τ weights synced from the het seed, exactly as the
original paired design), then the hidden τ_mem is **frozen** at the chosen fixed value. We assert no τ
parameter is trainable.

In [ ]:
# ========================== 4. Build frozen-tau hom ==========================
def build_fixed_hom(seed, tau_mem_ms):
    """Homogeneous net with hidden tau_mem FROZEN at tau_mem_ms; init weights = the het seed's (paired)."""
    # het seed init reproduces the original paired initial weights (deterministic from the seed);
    # its tau values are irrelevant — we only copy the NON-tau weights.
    het_init, _ = src.build_heterogeneous_model(TASK_KEY, seed)

    prms = src._base_prms_for_task(TASK_KEY, seed)
    prms.update({"het_ab": 0, "train_ab": 0, "train_hom_ab": 0})   # single, NON-trainable tau -> frozen
    prms.update({"dtype": torch.float, "device": DEVICE, "cuda": DEVICE.type == "cuda"})
    src.set_seed(seed)
    model = src.RSNN(prms, rec=True).to(DEVICE)
    src.sync_non_tau_parameters(het_init, model)                   # identical init weights to original hom

    # hidden layer: frozen tau_mem = fixed value, tau_syn = default 10 ms
    src.set_layer_taus_ms(model.network[0], tau_syn_ms=src.DEFAULT_HIDDEN_TAU_SYN_MS,
                          tau_mem_ms=float(tau_mem_ms), time_step=prms["time_step"])
    # output layer: defaults (frozen, as in the original)
    src.set_layer_taus_ms(model.network[1], tau_syn_ms=src.DEFAULT_OUTPUT_TAU_SYN_MS,
                          tau_mem_ms=src.DEFAULT_OUTPUT_TAU_MEM_MS, time_step=prms["time_step"])

    n_trainable_tau = sum(p.requires_grad for n, p in model.named_parameters() if ("alpha" in n or "beta" in n))
    assert n_trainable_tau == 0, "tau params must be frozen for the fixed-tau control!"
    # sanity: hidden tau actually equals the requested value, and is uniform
    got = src.decay_to_tau_ms(model.network[0].beta.detach().cpu().numpy().ravel(), prms["time_step"])
    assert np.allclose(got, tau_mem_ms, atol=1e-6) and np.unique(np.round(got, 6)).size == 1
    return model, prms

print("build_fixed_hom ready.")

## Part 1 · 5 — Load SHD into RAM (once)

In [ ]:
# ========================== 5. Data ==========================
train_cache, test_cache = src.load_default_caches()

## Part 1 · 6 — Train all 20 controls (resumable — existing checkpoints are skipped)

~30–60 min/seed on a T4 → a few hours for all 20. Safe to disconnect and re-run; finished nets are skipped.

In [ ]:
# ========================== 6. Train ==========================
import time, json
train_results = []
for tau in FIXED_TAUS_MS:
    tau_tag = f"{int(round(tau))}"
    for seed in SEEDS:
        out_path = CKPT_OUT / OUT_NAME.format(s=seed, tau=tau_tag)
        if out_path.exists():
            ck = torch.load(out_path, map_location="cpu", weights_only=False)
            train_results.append((tau, seed, ck["history"]["test_acc"][-1]))
            print(f"tau={tau_tag}ms seed {seed}: exists, skip (acc={ck['history']['test_acc'][-1]:.3f})")
            continue
        print(f"\n=== tau={tau_tag}ms seed {seed}: training (50 epochs, frozen tau_mem={tau} ms) ===")
        model, prms = build_fixed_hom(seed, tau)
        t0 = time.time()
        history = src.train_experiment(model, prms, train_cache, test_cache)
        elapsed = time.time() - t0
        src.save_model_checkpoint(
            out_path, model, history, elapsed, prms,
            extra={"label": out_path.stem, "task": TASK_KEY, "task_key": TASK_KEY,
                   "pair_seed": seed, "pair_role": "homogeneous_fixed_tau",
                   "run_label": "fixed_tau_sweep", "mem_distribution_family": MEM_FAMILY,
                   "control_type": "hom_tau_frozen_fixed", "fixed_tau_mem_ms": float(tau)},
        )
        train_results.append((tau, seed, history["test_acc"][-1]))
        print(f"   saved -> {out_path.name} | final test_acc={history['test_acc'][-1]:.3f} ({elapsed/60:.1f} min)")
print("\nPart 1 training complete.")

## Part 1 · 7 — Accuracy: het vs hom(~25 ms) vs hom(15 ms) vs hom(35 ms)

If the fixed-τ homs **don't** close the gap to het, the advantage is *diversity per se*. If a particular
single τ catches up, much of it was timescale-matching.

In [ ]:
# ========================== 7. Accuracy comparison ==========================
import matplotlib.pyplot as plt
def _acc(path):
    return float(torch.load(path, map_location="cpu", weights_only=False)["history"]["test_acc"][-1])

CONDS = ["het", "hom15", "hom25", "hom35"]
COND_PATH = {
    "het":   lambda s: REFCK / HET_NAME.format(s=s),
    "hom25": lambda s: REFCK / HOM_NAME.format(s=s),
    "hom15": lambda s: CKPT_OUT / OUT_NAME.format(s=s, tau="15"),
    "hom35": lambda s: CKPT_OUT / OUT_NAME.format(s=s, tau="35"),
}
acc = {c: [] for c in CONDS}
for s in SEEDS:
    for c in CONDS:
        p = COND_PATH[c](s)
        acc[c].append(_acc(p) if p.exists() else np.nan)
    print(f"seed {s}: " + "  ".join(f"{c}={acc[c][-1]:.3f}" for c in CONDS))

means = {c: np.nanmean(acc[c]) for c in CONDS}
stds  = {c: np.nanstd(acc[c], ddof=1) for c in CONDS}
print("\nMEANS  " + "  ".join(f"{c}={means[c]:.3f}" for c in CONDS))

labels = ["het", "hom\n15 ms", "hom\n~25 ms", "hom\n35 ms"]
order  = ["het", "hom15", "hom25", "hom35"]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.bar(labels, [means[c] for c in order], yerr=[stds[c] for c in order], capsize=5,
       color=["#d1495b", "#9bb7d4", "#30638e", "#3c6e47"], alpha=0.9)
for i, c in enumerate(order):
    if np.isfinite(means[c]): ax.text(i, means[c] + 0.01, f"{means[c]:.3f}", ha="center", fontweight="bold")
ax.axhline(0.25, color="gray", ls="--", lw=1, label="chance (25%)")
ax.set_ylabel("final test accuracy"); ax.set_ylim(0, max(v for v in means.values() if np.isfinite(v)) + 0.12)
ax.set_title("het vs fixed-τ homogeneous controls (10 seeds)", fontweight="bold")
ax.grid(alpha=0.3, axis="y"); ax.legend()
plt.tight_layout(); plt.savefig(OUTDIR / "fig_accuracy_tau_ladder.png", dpi=150, bbox_inches="tight"); plt.show()
print("saved ->", OUTDIR / "fig_accuracy_tau_ladder.png")

# Part 2 · Min/Max O-Information (Hardik's feedback)

Runs **after** Part 1. Three stages:
1. **Extract spikes** — forward each net through the SHD test set, cache the binary hidden-layer spikes
   (index-2 of the recurrent record), 4 ms-subsampled → `[neurons × T]`. (het / hom-25 from the bundled
   reference checkpoints; hom-15 / hom-35 from Part 1.)
2. **O/S sweep** — random k-subsets of the active neurons; per subset compute O- and S-information with
   Varley's `syntropy`.
3. **Min/Max O** + the **|Δτ| vs O** het scatter.

> **Dependency:** Part 2 needs `syntropy` (Thomas Varley's higher-order information library). The next cell
> tries to import it and, if missing, attempts a pip/git install. If that fails on Colab, copy your working
> `syntropy` package onto Drive and add it to `sys.path` (one line, noted in the cell).

In [ ]:
# ========================== 8. O-Info setup ==========================
import importlib
def _ensure_syntropy():
    try:
        importlib.import_module("syntropy.discrete"); return True
    except Exception:
        pass
    import subprocess, sys
    for spec in ["syntropy", "git+https://github.com/thosvarley/syntropy.git"]:
        try:
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", spec], check=True)
            importlib.import_module("syntropy.discrete"); return True
        except Exception:
            continue
    return False

if not _ensure_syntropy():
    # FALLBACK: copy your working syntropy package folder to Drive and point at its PARENT here, e.g.
    #   sys.path.insert(0, "/content/drive/MyDrive/pylibs"); import syntropy
    raise ImportError("Could not import/install `syntropy`. Add your local copy to sys.path (see comment).")

from syntropy.discrete.utils import construct_joint_distribution
from syntropy.discrete import o_information, s_information, total_correlation, dual_total_correlation
import hashlib, math, json, time, itertools
from scipy import stats
from data_gen import open_file

SPK_CACHE = OUTDIR / "spk_cache"; SPK_CACHE.mkdir(parents=True, exist_ok=True)
RECORDS   = OUTDIR / "os_info_records.jsonl"
FIGS      = OUTDIR
N_BATCHES = 8        # SHD-test batches per net (256 trials each) -> ~512k 4 ms time-bins
STRIDE    = 4        # 4 ms subsample (nb_steps=1000 @ 1 ms)
SUBSET_SIZES = [2, 4, 6, 8, 10, 12]
N_SUBSETS = 50
MIN_SUBSETS = 5
RK = [2, 4, 6, 8, 10, 12]
COND_COL = {"het": "#d1495b", "hom15": "#9bb7d4", "hom25": "#30638e", "hom35": "#3c6e47"}

def _ss(*p): return int(hashlib.sha256("|".join(map(str, p)).encode()).hexdigest()[:8], 16)

def os_info(sub_spk):
    """O,S,TC,DTC,n_states for a [k,T] binary spike matrix."""
    jd = construct_joint_distribution(np.asarray(sub_spk, dtype=int))
    return (float(o_information(jd)[1]), float(s_information(jd)[1]),
            float(total_correlation(jd)[1]), float(dual_total_correlation(jd)[1]), len(jd))

# sanity: XOR -> O<0 (synergy); 3 copies -> O>0 (redundancy)
_r = np.random.default_rng(0); _N = 5000
_X = _r.integers(0,2,_N); _Y = _r.integers(0,2,_N)
print("estimator check  XOR O=%+.3f (expect <0) | 3-copies O=%+.3f (expect >0)" % (
    os_info(np.array([_X, _Y, _X ^ _Y]))[0],
    os_info(np.array([_X, _X ^ (_r.random(_N)<0.05).astype(int), _X ^ (_r.random(_N)<0.05).astype(int)]))[0]))
print("Part 2 setup OK. records ->", RECORDS)

## Part 2 · Stage 1 — extract & cache spikes for all 40 nets (4 conditions × 10 seeds)

In [ ]:
# ========================== 9. Stage 1: spike extraction ==========================
@torch.no_grad()
def extract_spike_tensor(model, prms, units, times, labels, n_batches=N_BATCHES, stride=STRIDE):
    nb_hidden = int(model.network[0].output_size)
    nb_steps  = int(prms["nb_steps"]); nb_inputs = int(prms["nb_inputs"])
    batch_size = int(prms["batch_size"]); inv_dt = 1.0 / float(prms["time_step"])
    class_list = prms["class_list"]
    sample_idx = np.where(np.isin(labels, class_list))[0]
    n_batches = min(int(n_batches), -(-len(sample_idx) // batch_size))
    chunks = []
    for b in range(n_batches):
        bi = sample_idx[batch_size * b: min(len(sample_idx), batch_size * (b + 1))]
        abs_ = len(bi)
        t_arr = [np.round(times[i] * inv_dt).astype(np.int64) for i in bi]
        u_arr = [units[i] for i in bi]
        lens = np.array([len(a) for a in t_arr], dtype=np.int64)
        if lens.sum():
            ts = np.concatenate(t_arr); us = np.concatenate(u_arr)
            bc = np.repeat(np.arange(abs_, dtype=np.int64), lens)
            v = ts < nb_steps; ts, us, bc = ts[v], us[v], bc[v]
            idx = torch.from_numpy(np.stack([bc, ts, us]))
            x = torch.sparse_coo_tensor(idx, torch.ones(ts.size, dtype=torch.float32),
                                        torch.Size([abs_, nb_steps, nb_inputs])).to_dense()
        else:
            x = torch.zeros(abs_, nb_steps, nb_inputs)
        x = x.clamp(max=1.0).to(DEVICE)
        layer_recs = model(0, 0, x)
        spk = layer_recs[0][2][:, ::stride, :].detach().cpu().numpy()   # index 2 = binary spikes
        chunks.append(np.transpose(spk, (2, 0, 1)).reshape(nb_hidden, -1))
    return np.concatenate(chunks, axis=1)

def load_net(ckpt_path):
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    prms = dict(ck["prms"]); prms["dtype"] = torch.float
    prms["device"] = DEVICE; prms["cuda"] = DEVICE.type == "cuda"
    model = src.RSNN(prms, rec=True).to(DEVICE)
    model.load_state_dict(ck["model_state_dict"]); model.eval()
    return model, prms

# registry: cond -> (checkpoint path fn)
NETS = {c: COND_PATH[c] for c in CONDS}   # reuse Part 1's path map (het, hom15, hom25, hom35)

print("Loading SHD test set ...")
raw_u, raw_t, raw_l = open_file(str(src.SHD_TEST))
try:
    units = list(raw_u[:]); times = list(raw_t[:]); labels = np.array(raw_l[:])
finally:
    raw_u._v_file.close()
print(f"  test samples: {len(units)}")

for cond in CONDS:
    for seed in SEEDS:
        cache_path = SPK_CACHE / f"4class_seed{seed}_{cond}_spk.npy"
        if cache_path.exists():
            continue
        ckpt = NETS[cond](seed)
        if not ckpt.exists():
            print(f"{cond} seed {seed}: MISSING checkpoint {ckpt.name}"); continue
        model, prms = load_net(ckpt)
        spk = extract_spike_tensor(model, prms, units, times, labels)
        assert np.all((spk == 0) | (spk == 1)), "spike tensor not binary!"
        np.save(cache_path, spk.astype(np.uint8)); del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        print(f"{cond} seed {seed}: saved {spk.shape}  rate={spk.mean():.4f}  active={(spk.sum(1)>0).sum()}/{spk.shape[0]}")
print("Stage 1 complete.")

## Part 2 · Stage 2 — O/S over random k-subsets (resumable)

In [ ]:
# ========================== 10. Stage 2: O/S sweep ==========================
spk_cache = {}
for cond in CONDS:
    for seed in SEEDS:
        p = SPK_CACHE / f"4class_seed{seed}_{cond}_spk.npy"
        if not p.exists():
            print(f"{cond} seed {seed}: MISSING spike cache — run Stage 1"); continue
        spk = np.load(p).astype(int); active = np.where(spk.sum(1) > 0)[0]
        spk_cache[(cond, seed)] = spk[active]

done = set()
if RECORDS.exists():
    for line in open(RECORDS):
        try:
            r = json.loads(line); done.add((r["cond"], r["seed"], r["k"], r["subset_draw"]))
        except Exception:
            pass
print(f"Resuming: {len(done)} subset records present.")

rec_file = open(RECORDS, "a")
try:
    for (cond, seed), spk in spk_cache.items():
        n_active, T = spk.shape
        for k in SUBSET_SIZES:
            if k > n_active:
                continue
            cap = min(N_SUBSETS, math.comb(n_active, k))
            cap = max(cap, min(MIN_SUBSETS, math.comb(n_active, k)))
            rng = np.random.default_rng(_ss(cond, seed, k)); seen, subsets = set(), []
            while len(subsets) < cap:
                ss = tuple(sorted(rng.choice(n_active, size=k, replace=False).tolist()))
                if ss not in seen: seen.add(ss); subsets.append(ss)
            todo = [(i, ss) for i, ss in enumerate(subsets) if (cond, seed, k, i) not in done]
            if not todo:
                continue
            ratio = T / (2 ** k); flag = "  <-- UNDERSAMPLED" if ratio < 10 else ""
            print(f"{cond} seed {seed} k={k:2d}: {len(todo)}/{len(subsets)} | T/2^k={ratio:.0f}{flag}")
            for i, sub in todo:
                try:
                    O, S, TC, DTC, nstates = os_info(spk[list(sub)])
                    rec = {"cond": cond, "seed": seed, "k": k, "subset_draw": i,
                           "O_info": O, "S_info": S, "TC": TC, "DTC": DTC,
                           "n_states": nstates, "n_active": int(n_active), "status": "ok",
                           "neurons": [int(x) for x in sub]}
                except Exception as exc:
                    rec = {"cond": cond, "seed": seed, "k": k, "subset_draw": i, "status": f"error:{type(exc).__name__}"}
                rec_file.write(json.dumps(rec) + "\n"); rec_file.flush()
finally:
    rec_file.close()
print("Stage 2 complete.")

## Part 2 · Stage 3 — Min / Max O-information (the corrected statistic)

Per (network, k): **min O** (most synergistic subset), **max O** (most redundant), **spread = max − min**.
Compared across the τ-ladder. Also prints paired het-vs-hom25 significance at k=8 (the reliable ceiling;
`T/2^k ≳ 2000`).

In [ ]:
# ========================== 11. Stage 3a: min / max O ==========================
recs = [json.loads(l) for l in open(RECORDS)] if RECORDS.exists() else []
ok = [r for r in recs if r.get("status") == "ok"]
print(f"{len(recs)} records ({len(ok)} ok)")

def vals(cond, seed, k):
    return np.array([r["O_info"] for r in ok if r["cond"] == cond and r["seed"] == seed and r["k"] == k])

agg = {c: {st: {} for st in ["min", "max", "spread"]} for c in CONDS}
print(f"\n{'k':>3} | " + " | ".join(f"{c:>22}" for c in CONDS))
print(f"{'':>3} | " + " | ".join(f"{'minO   maxO   spread':>22}" for c in CONDS))
for k in RK:
    line = f"{k:>3} | "
    for c in CONDS:
        mins, maxs = [], []
        for s in SEEDS:
            v = vals(c, s, k)
            if v.size: mins.append(v.min()); maxs.append(v.max())
        mn, mx = np.array(mins), np.array(maxs)
        agg[c]["min"][k] = mn; agg[c]["max"][k] = mx; agg[c]["spread"][k] = mx - mn
        line += f"{(mn.mean() if mn.size else np.nan):+6.3f} {(mx.mean() if mx.size else np.nan):+6.3f} {((mx-mn).mean() if mn.size else np.nan):6.3f} | "
    print(line)

print("\n--- paired het vs hom(~25 ms) at k=8 (10 seeds) ---")
for st in ["min", "max", "spread"]:
    h, o = agg["het"][st].get(8), agg["hom25"][st].get(8)
    if h is None or o is None or h.size != o.size or h.size == 0:
        print(f"O {st:>6}: (insufficient data)"); continue
    dz = (h - o).mean() / (h - o).std(ddof=1)
    print(f"O {st:>6}: het={h.mean():+.3f} hom25={o.mean():+.3f} diff={h.mean()-o.mean():+.3f} "
          f"t p={stats.ttest_rel(h, o).pvalue:.3f} Wilcox p={stats.wilcoxon(h, o).pvalue:.3f} dz={dz:+.2f}")

# figure: min & max O vs k for each condition
fig, ax = plt.subplots(figsize=(10, 6))
for c in CONDS:
    mn = [agg[c]["min"][k].mean() if agg[c]["min"].get(k) is not None and agg[c]["min"][k].size else np.nan for k in RK]
    mx = [agg[c]["max"][k].mean() if agg[c]["max"].get(k) is not None and agg[c]["max"][k].size else np.nan for k in RK]
    ax.plot(RK, mx, color=COND_COL[c], lw=2, marker="^", label=f"{c} max O (most redundant)")
    ax.plot(RK, mn, color=COND_COL[c], lw=2, ls="--", marker="v", label=f"{c} min O (most synergistic)")
    ax.fill_between(RK, mn, mx, color=COND_COL[c], alpha=0.06)
ax.axhline(0, color="black", lw=1); ax.set_xlabel("subset size k"); ax.set_ylabel("O-information")
ax.set_title("Min / Max O-information across the τ-ladder\n(range = how far the most synergistic vs most redundant subsets sit)", fontweight="bold")
ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(FIGS / "fig_minmax_oinfo.png", dpi=150, bbox_inches="tight"); plt.show()
print("saved ->", FIGS / "fig_minmax_oinfo.png")

## Part 2 · Stage 3b — Timescale-diversity vs Info (het scatter)

For each het subset: x = mean pairwise **|Δτ_mem|** among its neurons (from the het checkpoint), y = its
O-info. Tests Hardik's hypothesis that more timescale-diverse subsets are **less redundant**. (Homogeneous
nets have a single τ → Δτ = 0, so this is het-specific.)

In [ ]:
# ========================== 12. Stage 3b: |delta-tau| vs O (het) ==========================
tau_ms = lambda b: src.decay_to_tau_ms(b, 1e-3)
def het_tau_active(seed):
    spk = np.load(SPK_CACHE / f"4class_seed{seed}_het_spk.npy"); active = np.where(spk.sum(1) > 0)[0]
    beta = torch.load(REFCK / HET_NAME.format(s=seed), map_location="cpu",
                      weights_only=False)["model_state_dict"]["network.0.beta"].numpy().ravel()
    return tau_ms(beta)[active]

pts = {6: ([], []), 8: ([], [])}
for seed in SEEDS:
    tau_a = het_tau_active(seed)
    for r in ok:
        if r["cond"] != "het" or r["seed"] != seed or r["k"] not in pts: continue
        ns = r.get("neurons")
        if not ns: continue
        taus = tau_a[np.array(ns)]
        dtau = np.mean([abs(taus[i] - taus[j]) for i, j in itertools.combinations(range(len(taus)), 2)])
        pts[r["k"]][0].append(dtau); pts[r["k"]][1].append(r["O_info"])

print("=== het: mean pairwise |delta-tau| vs subset O-info ===")
fig, ax = plt.subplots(figsize=(8.5, 6))
for k, mk in [(6, "o"), (8, "s")]:
    x = np.array(pts[k][0]); y = np.array(pts[k][1])
    if x.size < 3: print(f"k={k}: insufficient points"); continue
    pr = stats.pearsonr(x, y); sp = stats.spearmanr(x, y)
    ax.scatter(x, y, s=10, alpha=0.35, marker=mk,
               label=f"k={k} (Pearson r={pr.statistic:+.2f}, p={pr.pvalue:.1e}; Spearman ρ={sp.statistic:+.2f})")
    b, a0 = np.polyfit(x, y, 1); xx = np.linspace(x.min(), x.max(), 50); ax.plot(xx, a0 + b * xx, lw=2)
    print(f"k={k}: n={x.size}  Pearson r={pr.statistic:+.3f} (p={pr.pvalue:.2e})  Spearman rho={sp.statistic:+.3f} (p={sp.pvalue:.2e})")
ax.axhline(0, color="black", lw=1, alpha=0.5)
ax.set_xlabel("mean pairwise |Δτ_mem| among the subset's neurons (ms)")
ax.set_ylabel("subset O-information  (lower = more synergistic)")
ax.set_title("het: do more timescale-diverse subsets carry less-redundant information?", fontweight="bold")
ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.savefig(FIGS / "fig_delay_vs_info.png", dpi=150, bbox_inches="tight"); plt.show()
print("saved ->", FIGS / "fig_delay_vs_info.png")

## Done

**Outputs (under `outputs/`):**
- `checkpoints/4class_seed*_hom_fixed15ms_lognormal.pt`, `…_fixed35ms_…` — the 20 new controls.
- `fig_accuracy_tau_ladder.png` — het vs hom-15/25/35 accuracy.
- `os_info_records.jsonl` — per-subset O/S for all four conditions.
- `fig_minmax_oinfo.png` — min/max O across the τ-ladder.
- `fig_delay_vs_info.png` — het |Δτ| vs O scatter.

**To merge with the main project:** copy `outputs/checkpoints/*` into
`Project Files/Checkpoints/SeededRuns/seeded_run_1_64n_homfix/4class_lognormal/`, then re-run the corrected
PCA / PC-reconstruction dimensionality measures on the 15/25/35 ms ladder (WS5).